# 1주차 과제 - NSMC LSTM 감성 분석

- **데이터셋**: NSMC (네이버 영화 리뷰) — 학습 150,000건 / 테스트 50,000건
- **목표**: Bidirectional LSTM으로 감성 분류 구현, Test Accuracy **85% 이상** 달성

## 0. 환경 설정

In [ ]:
!pip install -q KoNLPy

In [ ]:
import numpy as np
import pandas as pd
import re
import time
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from konlpy.tag import Okt
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, Dense, LSTM, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# 재현성을 위한 시드 고정
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 1. 데이터 로드

In [ ]:
# NSMC 데이터 다운로드
DATA_TRAIN_PATH = tf.keras.utils.get_file(
    "ratings_train.txt",
    "https://raw.github.com/ironmanciti/Infran_NLP/main/data/naver_movie/ratings_train.txt"
)
DATA_TEST_PATH = tf.keras.utils.get_file(
    "ratings_test.txt",
    "https://raw.github.com/ironmanciti/Infran_NLP/main/data/naver_movie/ratings_test.txt"
)

In [ ]:
train_data = pd.read_csv(DATA_TRAIN_PATH, delimiter='\t')
test_data  = pd.read_csv(DATA_TEST_PATH,  delimiter='\t')

print("Train shape:", train_data.shape)
print("Test shape :", test_data.shape)
train_data.head()

## 2. 데이터 전처리

과제 가이드라인의 4단계 전처리 파이프라인을 순서대로 구현

### STEP 1 — 결측치 제거

In [ ]:
# NaN 제거
train_data.dropna(inplace=True)
test_data.dropna(inplace=True)

# 빈 문자열 제거
train_data = train_data[train_data['document'].str.strip() != '']
test_data  = test_data[test_data['document'].str.strip() != '']

print("결측치 제거 후")
print("Train:", train_data.shape, "| Test:", test_data.shape)
print("남은 null:")
print(train_data.isnull().sum())

### STEP 2 ~ 4 — 특수문자 제거 + 형태소 분석 + 길이 필터링

In [ ]:
okt = Okt()

# 허용할 품사 (명사, 동사, 형용사)
ALLOWED_POS = {'Noun', 'Verb', 'Adjective'}

def preprocessing(sentence):
    # STEP 2: 한글·공백만 남기고 특수문자 제거
    sentence = re.sub('[^가-힣 ]', '', str(sentence))
    
    # STEP 3: 형태소 분석 — Noun, Verb, Adjective만 추출
    morphs = okt.pos(sentence, stem=True)
    tokens = [word for word, pos in morphs if pos in ALLOWED_POS]
    
    return tokens  # 리스트 반환

In [ ]:
%%time

train_sentences, train_labels = [], []
test_sentences,  test_labels  = [], []

# 훈련 데이터 전처리
for i, (sent, label) in enumerate(zip(train_data['document'], train_data['label'])):
    if i % 10000 == 0:
        print(f"Train processed = {i}")
    tokens = preprocessing(sent)
    # STEP 4: 형태소 1개 이하 제거
    if len(tokens) > 1:
        train_sentences.append(tokens)
        train_labels.append(label)

# 테스트 데이터 전처리
for i, (sent, label) in enumerate(zip(test_data['document'], test_data['label'])):
    if i % 10000 == 0:
        print(f"Test processed = {i}")
    tokens = preprocessing(sent)
    if len(tokens) > 1:
        test_sentences.append(tokens)
        test_labels.append(label)

print(f"\n전처리 완료 — Train: {len(train_sentences)}건, Test: {len(test_sentences)}건")

In [ ]:
# 레이블을 numpy 배열로 변환
train_labels = np.array(train_labels)
test_labels  = np.array(test_labels)

print("Train labels shape:", train_labels.shape)
print("Test  labels shape:", test_labels.shape)
print("긍정(1) 비율 — train:", train_labels.mean().round(3),
      "| test:", test_labels.mean().round(3))

## 3. Tokenizer + Padding

- `num_words=30000` 
- `maxlen=100` 

In [ ]:
MAX_VOCAB = 30000
MAX_LEN   = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(train_sentences)

train_sequences = tokenizer.texts_to_sequences(train_sentences)
test_sequences  = tokenizer.texts_to_sequences(test_sentences)

print("어휘 사전 크기:", len(tokenizer.word_index))

In [ ]:
# 시퀀스 길이 분포 시각화
all_lens = [len(s) for s in train_sequences + test_sequences]
plt.figure(figsize=(8, 4))
plt.hist(all_lens, bins=40, color='steelblue', edgecolor='white')
plt.axvline(MAX_LEN, color='red', linestyle='--', label=f'maxlen={MAX_LEN}')
plt.title('시퀀스 길이 분포')
plt.xlabel('길이')
plt.ylabel('빈도')
plt.legend()
plt.tight_layout()
plt.show()

print(f"95th 퍼센타일 길이: {int(np.percentile(all_lens, 95))}")

In [ ]:
# Padding 적용
train_padded = pad_sequences(train_sequences, maxlen=MAX_LEN, padding='post', truncating='post')
test_padded  = pad_sequences(test_sequences,  maxlen=MAX_LEN, padding='post', truncating='post')

print("train_padded shape:", train_padded.shape)
print("test_padded shape :", test_padded.shape)

## 4. 모델 구현 — Bidirectional LSTM

In [ ]:
model = Sequential([
    # Embedding: (batch, MAX_LEN) → (batch, MAX_LEN, 128)
    Embedding(MAX_VOCAB, 128, input_length=MAX_LEN),

    # 1st Bidirectional LSTM: return_sequences=True → 다음 LSTM에 시퀀스 전달
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),

    # 2nd Bidirectional LSTM: 최종 hidden state만 반환
    Bidirectional(LSTM(32)),
    Dropout(0.3),

    # Dense 분류 헤드
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')   # 이진 분류 (긍정=1, 부정=0)
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. 모델 학습

In [ ]:
# 콜백 설정
callbacks = [
    # 성능이 plateau에 도달하면 학습률 감소
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    # val_loss 기준 최고 성능 모델 저장
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    # 5 epoch 동안 개선 없으면 조기 종료
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

In [ ]:
%%time

history = model.fit(
    train_padded, train_labels,
    epochs=20,
    batch_size=128,
    validation_data=(test_padded, test_labels),
    callbacks=callbacks,
    verbose=1
)

## 6. 학습 곡선 시각화

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy 곡선
ax1.plot(history.history['accuracy'],     label='Train Accuracy', color='steelblue')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy',   color='tomato', linestyle='--')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(alpha=0.3)

# Loss 곡선
ax2.plot(history.history['loss'],     label='Train Loss', color='steelblue')
ax2.plot(history.history['val_loss'], label='Val Loss',   color='tomato', linestyle='--')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('학습 곡선 (Bidirectional LSTM)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. 최종 성능 평가 (Test Accuracy)

In [ ]:
loss, accuracy = model.evaluate(test_padded, test_labels, verbose=0)
print(f"Test Loss    : {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}  ({accuracy*100:.2f}%)")

if accuracy >= 0.85:
    print("Test Accuracy 85% 이상")
else:
    print("하이퍼파라미터 조정 필요")

## 8. Confusion Matrix

In [ ]:
# 예측값 생성
y_pred_prob = model.predict(test_padded, verbose=0)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()
y_true = test_labels

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['부정(0)', '긍정(1)'],
            yticklabels=['부정(0)', '긍정(1)'])
plt.xlabel('예측값')
plt.ylabel('실제값')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# 상세 리포트
print("\n[Classification Report]")
print(classification_report(y_true, y_pred, target_names=['부정', '긍정']))

## 9. 샘플 예측 테스트

In [ ]:
def predict_sentiment(texts):
    """리뷰 텍스트 리스트를 받아 긍정/부정 예측 결과 반환"""
    preprocessed = [preprocessing(t) for t in texts]
    seqs    = tokenizer.texts_to_sequences(preprocessed)
    padded  = pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')
    probs   = model.predict(padded, verbose=0).flatten()

    for text, prob in zip(texts, probs):
        label = '긍정 😊' if prob >= 0.5 else '부정 😞'
        print(f"[{label}] ({prob:.3f})  {text}")

sample_reviews = [
    "이 영화는 정말 감동적이고 배우들의 연기가 훌륭했다",
    "스토리가 너무 지루하고 억지스러워서 보다가 잠들었다",
    "오랜만에 본 수작이다 다시 보고 싶은 영화",
    "돈 아깝고 시간 낭비 최악이었음",
    "배우들 연기는 괜찮았지만 결말이 아쉬웠다"
]

predict_sentiment(sample_reviews)

## 10. (선택) 하이퍼파라미터 실험 비교

- 실험 A: `MAX_LEN=100, LSTM(64,32), Dropout=0.3` ← 메인 모델
- 실험 B: `MAX_LEN=50,  LSTM(128,64), Dropout=0.4`

In [ ]:
def build_model(vocab_size, max_len, lstm1, lstm2, dropout):
    m = Sequential([
        Embedding(vocab_size, 128, input_length=max_len),
        Bidirectional(LSTM(lstm1, return_sequences=True)),
        Dropout(dropout),
        Bidirectional(LSTM(lstm2)),
        Dropout(dropout),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

# 실험 B 모델
MAX_LEN_B = 50
train_padded_b = pad_sequences(train_sequences, maxlen=MAX_LEN_B, padding='post', truncating='post')
test_padded_b  = pad_sequences(test_sequences,  maxlen=MAX_LEN_B, padding='post', truncating='post')

model_b = build_model(MAX_VOCAB, MAX_LEN_B, lstm1=128, lstm2=64, dropout=0.4)

history_b = model_b.fit(
    train_padded_b, train_labels,
    epochs=10, batch_size=128,
    validation_data=(test_padded_b, test_labels),
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=1
)

_, acc_a = model.evaluate(test_padded,   test_labels, verbose=0)
_, acc_b = model_b.evaluate(test_padded_b, test_labels, verbose=0)

print(f"\n실험 A (MAX_LEN=100, LSTM 64/32, Dropout 0.3): {acc_a*100:.2f}%")
print(f"실험 B (MAX_LEN=50,  LSTM 128/64, Dropout 0.4): {acc_b*100:.2f}%")

# 결과 시각화
plt.figure(figsize=(6, 4))
bars = plt.bar(['실험 A\n(메인)', '실험 B\n(비교)'], [acc_a, acc_b],
               color=['steelblue', 'salmon'], width=0.5)
plt.axhline(0.85, color='red', linestyle='--', label='목표 85%')
for bar, acc in zip(bars, [acc_a, acc_b]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{acc*100:.2f}%', ha='center', fontsize=11)
plt.ylim(0.80, 0.95)
plt.ylabel('Test Accuracy')
plt.title('하이퍼파라미터 실험 비교')
plt.legend()
plt.tight_layout()
plt.show()